# BERT Fine-Tuning for Product Review Sentiment Analysis

**Course:** UIR S8 — Integrated Project  
**Deliverable:** Week 2 (W2)  
**Author:** Anys Louzal

This notebook is the Week 2 deliverable of the *product-review-intelligence* project.
It fine-tunes `bert-base-uncased` on the McAuley-Lab Amazon Reviews 2023 dataset
(category *All_Beauty*) for **3-class sentiment classification**
(negative / neutral / positive).

### Pipeline at a glance
1. Install dependencies and import libraries.
2. Load 10 000 reviews, map star ratings to 3 sentiment classes, inspect the data.
3. Tokenize with the BERT tokenizer (max 128 tokens) and split 80 / 20.
4. Build a `BertForSequenceClassification` head and a HuggingFace `Trainer`.
5. Train for 3 epochs and plot the loss curves.
6. Evaluate: classification report, confusion matrix, qualitative examples.
7. Persist the model (HF format + raw `state_dict`).
8. Expose a `predict_sentiment(text)` inference function and smoke-test it.

> **Runtime:** designed for Google Colab with a free T4 GPU.
> On CPU the training step will take much longer — switch runtime to GPU
> before running Section 5.

## 1. Setup & imports

We install everything we need in a single cell so the notebook runs
top-to-bottom on a fresh Colab VM. `transformers` gives us BERT and the
`Trainer` loop, `datasets` handles the HuggingFace corpus, and
`scikit-learn` supplies the evaluation metrics and the train/val split.

In [ ]:
# Install dependencies. On Colab this is idempotent — subsequent runs are fast.
!pip install -q transformers==4.44.2 datasets==2.20.0 torch scikit-learn matplotlib seaborn accelerate

In [ ]:
# ---- Standard library ------------------------------------------------
import os
import json
import random
from pathlib import Path

# ---- Numerical / data ------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- PyTorch ---------------------------------------------------------
import torch
from torch.utils.data import Dataset as TorchDataset

# ---- HuggingFace stack ----------------------------------------------
from datasets import load_dataset, Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

# ---- Metrics --------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Reproducibility: seed everything we can.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch: {torch.__version__}  |  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load & explore the dataset

We pull the first **10 000 rows** of `McAuley-Lab/Amazon-Reviews-2023`
(config `raw_review_All_Beauty`) in streaming mode so we never
materialise the full corpus. Then we map star ratings to three
sentiment classes:

| Rating | Sentiment | Label id |
|--------|-----------|----------|
| 1 – 2  | negative  | 0 |
| 3      | neutral   | 1 |
| 4 – 5  | positive  | 2 |

This collapses a noisy 5-way regression problem into a cleaner 3-way
classification problem that is defensible in an oral presentation.

In [ ]:
N_ROWS = 10_000

stream = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
rows = list(stream.take(N_ROWS))
raw_df = pd.DataFrame(rows)
print(f"Rows pulled: {len(raw_df):,}")
raw_df.head(3)

In [ ]:
# Defensive column lookup — the dataset schema occasionally drifts.
rating_col = next((c for c in ["rating", "overall", "stars"] if c in raw_df.columns), None)
text_col = next((c for c in ["text", "reviewText", "review_body"] if c in raw_df.columns), None)
assert rating_col and text_col, f"Unexpected schema: {list(raw_df.columns)}"

def rating_to_label(rating: float) -> int:
    """Map a 1–5 star rating to {0: negative, 1: neutral, 2: positive}."""
    r = int(round(rating))
    if r <= 2:
        return 0
    if r == 3:
        return 1
    return 2

df = raw_df[[rating_col, text_col]].copy()
df.columns = ["rating", "text"]
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating", "text"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() > 0]
df["label"] = df["rating"].apply(rating_to_label)

LABEL_NAMES = ["negative", "neutral", "positive"]
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in ID2LABEL.items()}

print(f"Usable rows: {len(df):,}")
df.head()

In [ ]:
# Class distribution — this is the most important sanity check before
# training. Heavy imbalance will bias both the model and the F1 metric.
class_counts = df["label"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([LABEL_NAMES[i] for i in class_counts.index], class_counts.values,
       color=["#d9534f", "#f0ad4e", "#5cb85c"])
ax.set_title("Class distribution (10 000 All_Beauty reviews)")
ax.set_ylabel("Number of reviews")
for i, v in enumerate(class_counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout(); plt.show()
class_counts

In [ ]:
# Qualitative check: two samples per class.
for lbl in sorted(df["label"].unique()):
    name = LABEL_NAMES[lbl]
    print(f"\n===== {name.upper()} =====")
    for i, row in enumerate(df[df["label"] == lbl].sample(2, random_state=SEED).itertuples(), 1):
        txt = row.text.replace("\n", " ")
        print(f"  [{i}] ({row.rating}★) {txt[:220]}{'…' if len(txt) > 220 else ''}")

## 3. Tokenization

BERT expects subword token ids. We use the fast tokenizer for
`bert-base-uncased` with **max length 128** — enough to cover the vast
majority of Amazon reviews while keeping memory in check so the free
T4 GPU can handle batch size 16.

We then split 80 / 20 with stratification so every class is represented
in both train and validation.

In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = BertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)

# Stratified 80/20 split at the pandas level — keeps class proportions
# identical between train and val.
train_df, val_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,}    Val: {len(val_df):,}")

In [ ]:
# Wrap each split in a HuggingFace Dataset, then map the tokenizer over
# it. `batched=True` is much faster than element-wise tokenization.
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

# Trainer expects tensors on the target device; the collator handles
# dynamic padding per batch (saves GPU memory vs static padding to 128).
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_ds)

## 4. Model

`BertForSequenceClassification` wraps the base BERT encoder with a
linear classification head over the `[CLS]` token. We set
`num_labels=3` and pass the id↔label maps so the saved model knows
its own class names when it is reloaded for inference.

`TrainingArguments` uses the exact settings required by the brief.

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)

OUTPUT_DIR = "./bert_sentiment"

# `evaluation_strategy` is the pre-4.46 name; it still works in 4.44.2
# which we pinned above. Newer versions renamed it to `eval_strategy`.
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="none",          # silence wandb/tensorboard autodetection
    seed=SEED,
)
print(f"Output dir: {OUTPUT_DIR}")

## 5. Training

`compute_metrics` is called once per evaluation epoch. We return
**accuracy** and **weighted F1** — F1 weighted by class support is the
sensible default when classes are imbalanced (which ours will be:
Amazon skews heavily positive).

In [ ]:
def compute_metrics(eval_pred):
    """Accuracy + weighted F1 for Trainer's evaluation loop."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

In [ ]:
# Pull per-epoch train/val loss out of Trainer's log history and plot.
history = trainer.state.log_history
train_losses = [(h["epoch"], h["loss"]) for h in history if "loss" in h and "eval_loss" not in h]
val_losses = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(7, 4))
if train_losses:
    xs, ys = zip(*train_losses)
    ax.plot(xs, ys, label="train loss", marker="o", alpha=0.7)
if val_losses:
    xs, ys = zip(*val_losses)
    ax.plot(xs, ys, label="val loss", marker="s", linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training vs validation loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Evaluation

Three complementary views:

- **Classification report** — precision / recall / F1 per class.
- **Confusion matrix** — where the model is actually making mistakes.
- **Qualitative sample** — three correct and three wrong predictions
  so we can eyeball the failure modes.

In [ ]:
predictions = trainer.predict(val_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix (validation set)")
plt.tight_layout(); plt.show()

In [ ]:
# Reattach the original text to each validation row so we can print
# interpretable examples (val_ds has text removed).
val_texts = val_df["text"].tolist()
correct_idx = np.where(y_pred == y_true)[0]
wrong_idx = np.where(y_pred != y_true)[0]

rng = np.random.default_rng(SEED)
correct_sample = rng.choice(correct_idx, size=min(3, len(correct_idx)), replace=False)
wrong_sample = rng.choice(wrong_idx, size=min(3, len(wrong_idx)), replace=False)

print("===== 3 CORRECT predictions =====")
for i in correct_sample:
    print(f"  true={LABEL_NAMES[y_true[i]]:<8}  pred={LABEL_NAMES[y_pred[i]]:<8}  | {val_texts[i][:200]}")

print("\n===== 3 WRONG predictions =====")
for i in wrong_sample:
    print(f"  true={LABEL_NAMES[y_true[i]]:<8}  pred={LABEL_NAMES[y_pred[i]]:<8}  | {val_texts[i][:200]}")

## 7. Save the model

We persist the artefacts in **two** formats:

1. **HuggingFace format** (`save_pretrained`) — folder with config,
   tokenizer, and weights; reloadable with
   `from_pretrained(path)` anywhere.
2. **Raw PyTorch state_dict** (`model.pt`) — smaller, portable, usable
   from plain PyTorch without the `transformers` save format.

In [ ]:
SAVE_DIR = Path("./models/bert_sentiment")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# HuggingFace format (folder with config.json, tokenizer.json, weights).
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))

# Raw PyTorch weights.
torch.save(model.state_dict(), SAVE_DIR / "model.pt")

# Final metrics report — useful to copy-paste into the project log.
final_acc = accuracy_score(y_true, y_pred)
final_f1 = f1_score(y_true, y_pred, average="weighted")
print(f"Saved model to: {SAVE_DIR.resolve()}")
print(f"Final accuracy     : {final_acc:.4f}")
print(f"Final F1 (weighted): {final_f1:.4f}")

with open(SAVE_DIR / "metrics.json", "w") as fp:
    json.dump({"accuracy": final_acc, "f1_weighted": final_f1}, fp, indent=2)

## 8. Inference function

A small helper that loads the saved model on first call and exposes
the clean dictionary interface the downstream CrewAI agent expects:

```python
{
  "label": "positive",
  "confidence": 0.987,
  "scores": {"positive": 0.987, "neutral": 0.010, "negative": 0.003}
}
```

In [ ]:
_MODEL_CACHE = {"model": None, "tokenizer": None}

def _load_saved(path: str = str(SAVE_DIR)):
    """Lazy-load the fine-tuned model + tokenizer once per session."""
    if _MODEL_CACHE["model"] is None:
        tok = BertTokenizerFast.from_pretrained(path)
        mdl = BertForSequenceClassification.from_pretrained(path)
        mdl.to(DEVICE)
        mdl.eval()
        _MODEL_CACHE["model"] = mdl
        _MODEL_CACHE["tokenizer"] = tok
    return _MODEL_CACHE["model"], _MODEL_CACHE["tokenizer"]

def predict_sentiment(text: str) -> dict:
    """Classify a single review string.

    Returns a dict with the top label, its confidence, and the full
    softmax distribution over the three classes.
    """
    mdl, tok = _load_saved()
    enc = tok(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = mdl(**enc).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()

    top_id = int(np.argmax(probs))
    return {
        "label": LABEL_NAMES[top_id],
        "confidence": float(probs[top_id]),
        "scores": {name: float(probs[i]) for i, name in enumerate(LABEL_NAMES)},
    }

In [ ]:
# Smoke-test on three hand-crafted reviews (positive / negative / mixed).
EXAMPLES = [
    "Absolutely love this moisturizer! My skin feels hydrated all day and the scent is lovely.",
    "Terrible experience — the bottle arrived half-empty and the product broke me out within two days.",
    "The packaging is gorgeous and the shade range is impressive, but the formula is way too drying on my skin.",
]

for review in EXAMPLES:
    result = predict_sentiment(review)
    print(f"\nREVIEW: {review}")
    print(json.dumps(result, indent=2))

---

### Hand-off to Week 3

The folder `./models/bert_sentiment/` now contains everything the W3
pipeline needs:

- `config.json`, `tokenizer.json`, `vocab.txt`, model weights — the
  HuggingFace-format checkpoint.
- `model.pt` — the raw PyTorch `state_dict`.
- `metrics.json` — the final accuracy / F1 for the report.

In Week 3 we will wrap `predict_sentiment` inside a CrewAI `BaseTool`
so the orchestrator agent can call BERT locally instead of relying on
the LLM for sentiment classification.